# Part I: Construct SPX and NDX Futures Fair Values

* **SPX** and **NDX** futures expire on the third Friday of March, June, September, and December.

In this assignment, **you are to price all futures expiring before the end of 2026.**

1. **Data gathering:** In order to construct futures fair value for a specific expiry, we need dividend yield as well as spot risk free interest rate to the futures expiry
    1. *Yield expectation:*
       Use "indicated yield" of corresponding ETFs, **SPY and QQQ, as proxy for SPX and NDX**. You can find them on Bloomberg terminal using `SPY <equity> DES <go>`
    2. *Spot risk free interest rate*
    3. You can use zero coupon treasury strips as a proxy for this purpose. You can find them on Bloomberg terminal using `FIT S/0-5 <go>` for 0 to 5-year maturities. It’s up to you to use either “Coupon Strips” or “Principal Strips” curves.

2. **Programming:**
    1. Write a python function to auto-generate the futures expiry dates
    2. Write a python function to look-up the spot risk free interest rate that has nearest maturity after a given date
    3. Write a python function to calculate the futures fair value, assuming interest rate curve is static

3. **Output results:**
   Calculate and output futures fair values given the spot index SPX=6600.35 and NDX=24223.69 in a table in the follow format

| | DividendYld | SpotRate | SPX | NDX |
|---|---|---|---|---|
| Spot | | | 6600.35 | 24223.69 |
| Dec25 | | | | |
| Mar26 | | | | |
| Jun26 | | | | |
| Sep26 | | | | |
| Dec26 | | | | |

In [604]:
import pandas as pd
import pytz
from datetime import datetime, timedelta
import datetime
import math

## 1. **Data gathering:** In order to construct futures fair value for a specific expiry, we need dividend yield as well as spot risk free interest rate to the futures expiry

- a. *Yield expectation:*
   Use "indicated yield" of corresponding ETFs, SPY and QQQ, as proxy. You can find them on Bloomberg terminal using `SPY <equity> DES <go>`
- b. *Spot risk free interest rate*
- c. You can use zero coupon treasury strips as a proxy for this purpose. You can find them on Bloomberg terminal using `FIT S/0-5 <go>` for 0 to 5-year maturities. It’s up to you to use either “Coupon Strips” or “Principal Strips” curves.

____
### 1a. *Yield expectation:*
   Use "indicated yield" of corresponding ETFs, SPY and QQQ, as proxy. You can find them on Bloomberg terminal using `SPY <equity> DES <go>`

In [607]:
etf_futures_1m_data = pd.read_csv('etf_futures_1m_data.csv')
etf_futures_1m_data

,Datetime,SPY,QQQ,ES=F,NQ=F
0,2025-09-18 13:30:00+00:00,661.409973,595.070007,6681.75,24656.00
1,2025-09-18 13:31:00+00:00,662.090027,595.099976,6688.75,24686.50
2,2025-09-18 13:32:00+00:00,661.830017,595.184998,6686.00,24690.75
3,2025-09-18 13:33:00+00:00,661.890015,595.325012,6686.75,24695.50
4,2025-09-18 13:34:00+00:00,661.989990,595.119995,6687.75,24698.00
...,...,...,...,...,...
2717,2025-09-26 19:55:00+00:00,661.590027,595.889221,6695.00,24720.25
2718,2025-09-26 19:56:00+00:00,661.445007,595.780029,6693.25,24716.00
2719,2025-09-26 19:57:00+00:00,661.840027,596.119995,6697.50,24729.50
2720,2025-09-26 19:58:00+00:00,661.885010,596.150024,6698.00,24730.75


To obtain the Dividend Yield I went to the school's Bloomberg terminal and obtained the following:

- Net Dividend Yield for SPX is 1.1% as seen by the Net Indicated Yield
![My Image](https://raw.githubusercontent.com/SGhuman123/PhotoEdits/924bd6c21da8fac32b8d5a7a6552cc982fbe1096/Random/B06DCF05-E26D-41D3-85A9-67E3CBDFC329.JPG)

- Dividend Yield for QQQ is 0.46% as seen by the Net Indicated Yield

![My Image](https://raw.githubusercontent.com/SGhuman123/PhotoEdits/924bd6c21da8fac32b8d5a7a6552cc982fbe1096/Random/F371BABE-2FB2-4BFC-950E-FD68E51FCD01.JPG)

- 1b. *Spot risk free interest rate*
- 1c. You can use zero coupon treasury strips as a proxy for this purpose. You can find them on Bloomberg terminal using `FIT S/0-5 <go>` for 0 to 5-year maturities. It’s up to you to use either “Coupon Strips” or “Principal Strips” curves.

For me I decided to go along with the Coupon Strips instead of Principal Strips.

In [612]:
coupon_strips = pd.read_csv('CouponStrips.csv')
coupon_strips.head()

,#,Security,Ask,Bid,Mid,Chg
0,31,S 8/25,NaN,NaN,NaN,NaN
1,32,S N/25,3.991,3.888,3.9395,NaN
2,33,S 2/26,3.696,3.578,3.6370,0.009
3,34,S 5/26,3.732,3.605,3.6685,0.026
4,35,S 8/26,3.577,3.505,3.5410,-0.006


## 2. **Programming:**

- a. Write a python function to auto-generate the futures expiry dates
- b. Write a python function to look-up the spot risk free interest rate that has nearest maturity after a given date
- c. Write a python function to calculate the futures fair value, assuming interest rate curve is static

### 2a. Write a python function to auto-generate the futures expiry dates

* Futures contracts typically expire on the **third Friday** of each **calendar quarter**, making these dates crucial for traders to manage their positions effectively.
* We also need to know the **Yield expectation** in which we use the "indicated yield" of corresponding ETFs, **SPY and QQQ, as proxy for SPX and NDX futures**.
* We also use **zero coupon treasury strips** as a proxy for the **Spot risk free interest rate**.

In [615]:
etf_futures_1m_data.head()

,Datetime,SPY,QQQ,ES=F,NQ=F
0,2025-09-18 13:30:00+00:00,661.409973,595.070007,6681.75,24656.00
1,2025-09-18 13:31:00+00:00,662.090027,595.099976,6688.75,24686.50
2,2025-09-18 13:32:00+00:00,661.830017,595.184998,6686.00,24690.75
3,2025-09-18 13:33:00+00:00,661.890015,595.325012,6686.75,24695.50
4,2025-09-18 13:34:00+00:00,661.989990,595.119995,6687.75,24698.00


In [616]:
etf_futures_1m_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2722 entries, 0 to 2721
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   Datetime  2722 non-null   object 
 1   SPY       2722 non-null   float64
 2   QQQ       2722 non-null   float64
 3   ES=F      2722 non-null   float64
 4   NQ=F      2722 non-null   float64
dtypes: float64(4), object(1)
memory usage: 106.5+ KB


____
*The futures expiry dates are:*
- third Friday of March
- third Friday of June
- third Friday of Septemeber
- third Friday of December

So to determine the futures expiry date we need to generate a functions that takes in a pandas dataframe. The function would then have to look at the `Datetime` column and then subsequently look at the nearest futures expiry date. 

In [618]:
def third_friday(year, month):
    """Return datetime.date for monthly option expiration given year and
    month
    """
    # The 15th is the lowest third day in the month
    third = datetime.date(year, month, 15)
    # What day of the week is the 15th?
    w = third.weekday()
    # Friday is weekday 4
    if w != 4:
        # Replace just the day (of month)
        third = third.replace(day=(15 + (4 - w) % 7))
    third = datetime.datetime(third.year, third.month, third.day, tzinfo=pytz.UTC) + timedelta(days=1, microseconds=-1)
    return third.strftime("%Y-%m-%d")

Let us test our function.

In [620]:
sample_input = 'Dec25'

def monthToNum(shortMonth):
    return {
            'Mar': 3,
            'Jun': 6,
            'Sep': 9, 
            'Dec': 12
    }[shortMonth]

month = int(monthToNum(sample_input[:3]))
year = int('20'+sample_input[3:])

third_friday(year, month)

'2025-12-19'

Now let us gather a list of future expiry dates by considering the structure of our final output which is:

| | DividendYld | SpotRate | SPX | NDX |
|---|---|---|---|---|
| Spot | | | 6600.35 | 24223.69 |
| Dec25 | | | | |
| Mar26 | | | | |
| Jun26 | | | | |
| Sep26 | | | | |
| Dec26 | | | | |

In [622]:
tuple_of_inputs = ('Dec25', 'Mar26', 'Jun26', 'Sep26', 'Dec26')
list_of_future_expiry_dates = []
for i in tuple_of_inputs:
    month = int(monthToNum(i[:3]))
    year = int('20'+i[3:])

    list_of_future_expiry_dates.append(third_friday(year, month))

list_of_future_expiry_dates

['2025-12-19', '2026-03-20', '2026-06-19', '2026-09-18', '2026-12-18']

### 2b. Write a python function to look-up the spot risk free interest rate that has nearest maturity after a given date

* We also use **zero coupon treasury strips** as a proxy for the **Spot risk free interest rate**.

In [624]:
coupon_strips.head()

,#,Security,Ask,Bid,Mid,Chg
0,31,S 8/25,NaN,NaN,NaN,NaN
1,32,S N/25,3.991,3.888,3.9395,NaN
2,33,S 2/26,3.696,3.578,3.6370,0.009
3,34,S 5/26,3.732,3.605,3.6685,0.026
4,35,S 8/26,3.577,3.505,3.5410,-0.006


As we use the coupon strips as a proxy of the Spot risk free interest rate. Let us consider using the values in the column of Security `coupon_strips` as a proxy for the maturity dates. So there's this pattern in `coupon_strips` dataframe of:
- `S 2/Year` -> February
- `S 5/Year` -> May
- `S 8/Year` -> August
- `S N/Year` -> November

Since the specific day of the month isn't given, let us assume that the maturity date falls on the first day of the month. Also the first row where the `Ask`, `Bid`, `Mid` and `Chg` values are all NaN. Since we are only interested in the `Mid` to determine the spot free interest rate. Let us only obtain rows where Mid has no NaN values.

In [626]:
coupon_strips_cleaned = coupon_strips[coupon_strips['Mid'].notna()]
coupon_strips_cleaned.head()

,#,Security,Ask,Bid,Mid,Chg
1,32,S N/25,3.991,3.888,3.9395,NaN
2,33,S 2/26,3.696,3.578,3.6370,0.009
3,34,S 5/26,3.732,3.605,3.6685,0.026
4,35,S 8/26,3.577,3.505,3.5410,-0.006
5,36,S N/26,3.684,3.606,3.6450,-0.001


___
Let's examine how our input data would look like.

In [628]:
list_of_future_expiry_dates

['2025-12-19', '2026-03-20', '2026-06-19', '2026-09-18', '2026-12-18']

#### How I derived the function for 2b

In [630]:
# # What needs to be done
# # Input: date

# # Operations:
# # Check out all values of coupon_strips, 'Security' column
# # Create a dictionary to track the maturity dates of specific treasury strip and coupon rates
# # Function has to be such that the first Treasury strip that matures after your futures expiry 
# # Next, use that rate that is picked out and identified
# # We shall use the 'Mid' value as the spot free interest rate

# # Output: 
# # Spot risk free interest rate

# maturity_dates_and_spot_free_IR = {}
# coupon_number = 1

# for index, row in coupon_strips_cleaned.iterrows():

#     # Obtain month and year of maturity date
#     month = row['Security'][2:3]
#     year = '20'+row['Security'][4:36]

#     # Convert to integers  
#     year = int(year)
#     if month.isdigit() == False:
#         month = 11
#     else:
#         month = int(month)

#     # As we assume that the maturity date falls on the first day of the month
#     maturity_date = datetime.datetime(int(year), int(month), 1)
    
#     # Add to dictionary:
#     maturity_dates_and_spot_free_IR.update({coupon_number:[maturity_date, row['Mid']]})

#     coupon_number += 1

# maturity_dates_and_spot_free_IR

In [631]:
# # # Recap of our futures expiry dates
# list_of_future_expiry_dates[0]

In [632]:
# list_of_future_expiry_dates

In [633]:
# for date in list_of_future_expiry_dates:
#     confirmed_spot_free_IR = 0
    
#     # Futures expiry dates is in string format so we need to convert it to datetime format for easy comparison
#     future_expiry_dates = datetime.datetime(int(date[:4]), int(date[5:7]), int(date[8:10]))
#     # print(future_expiry_dates)
    
#     # Loop through dictionary with maturity dates and spot free IR
#     for key, value in maturity_dates_and_spot_free_IR.items():

#         # Obtain the maturity date
#         maturity_date = value[0]

#         # Obtain the risk free IR
#         spot_risk_free_IR = value[1]
        
#         # print("maturity_dates_and_spot_free_IR:", maturity_date, spot_risk_free_IR)

#         # Compare the futures expiry date with the maturity date
#         if future_expiry_dates <= maturity_date:
#             confirmed_spot_free_IR = spot_risk_free_IR

#             # Once we found the futures expiry date with 
#             # the nearest maturity in the future
#             # choose that as the spot free IR and we can end the loop
#             break
#         else:
#             continue            
#     print(future_expiry_dates, confirmed_spot_free_IR)

In [634]:
# list_of_future_expiry_dates

In [635]:
# maturity_dates_and_spot_free_IR

### 2b. Write a python function to look-up the spot risk free interest rate that has nearest maturity after a given date

* We also use **zero coupon treasury strips** as a proxy for the **Spot risk free interest rate**.

In [637]:
# What needs to be done
# Input: date

# Operations:
# Check out all values of coupon_strips, 'Security' column
# Create a new dictionary to track the maturity dates of specific treasury strip
# Function has to be such that the first Treasury strip that matures after your futures expiry 
# Next, use that rate that is picked out and identified
# We shall use the 'Mid' value as the spot free interest rate

# Output: 
# Spot risk free interest rate

maturity_dates_and_spot_free_IR = {}
coupon_number = 1

# Check out all values of coupon_strips, 'Security' column
# Create a new dictionary to track the maturity dates of specific treasury strip
for index, row in coupon_strips_cleaned.iterrows():

    # Obtain month and year of maturity date
    month = row['Security'][2:3]
    year = '20'+row['Security'][4:36]

    # Convert to integers  
    year = int(year)
    if month.isdigit() == False:
        month = 11
    else:
        month = int(month)

    # As we assume that the maturity date falls on the first day of the month
    maturity_date = datetime.datetime(int(year), int(month), 1)
    
    # Add to dictionary:
    maturity_dates_and_spot_free_IR.update({coupon_number:[maturity_date, row['Mid']]})

    coupon_number += 1

def look_up_spot_rf_rate(date):
    confirmed_spot_free_IR = 0
    
    # Futures expiry dates is in string format so we need to convert it to datetime format for easy comparison
    future_expiry_dates = datetime.datetime(int(date[:4]), int(date[5:7]), int(date[8:10]))

    # Loop through dictionary with maturity dates and spot free IR
    for key, value in maturity_dates_and_spot_free_IR.items():

        # Obtain the maturity date
        maturity_date = value[0]

        # Obtain the risk free IR
        spot_risk_free_IR = value[1]
        
        # print("maturity_dates_and_spot_free_IR:", maturity_date, spot_risk_free_IR)

        # Compare the futures expiry date with the maturity date
        if future_expiry_dates <= maturity_date:
            confirmed_spot_free_IR = spot_risk_free_IR

            # Once we found the futures expiry date with 
            # the nearest maturity in the future
            # choose that as the spot free IR and we can end the loop
            break
        else:
            continue            
    
    return confirmed_spot_free_IR

In [638]:
# Test our function out
# sample_date = list_of_future_expiry_dates[0]
# print(sample_date, look_up_spot_rf_rate(sample_date))

spot_free_interest_rates_after_given_dates = list(map(look_up_spot_rf_rate, list_of_future_expiry_dates))
spot_free_interest_rates_after_given_dates

[3.637, 3.6685, 3.541, 3.645, 3.5535]

### 2c. Write a python function to calculate the futures fair value, assuming interest rate curve is static

**Note to self:** Why do we need to assume interest rate curve is static?

**Ans:** This is because formula below assumes constant interest rates and dividend yields. For practical purposes, futures prices are often assumed to be
identical to forward prices.

#### Futures Pricing Formula

$$F_0 = S_0 * e^{(r-q)T}$$

* **$F_0$**: Fair value of the futures contract today.
* **$S_0$**: Spot price of the underlying index today.
* **$e$**: The base of the natural logarithm (~2.718).
* **$r$**: The continuously compounded risk-free interest rate.
* **$q$**: The continuously compounded dividend yield of the index.
* **$T$**: The time to maturity of the contract, expressed in years.


#### How I derived the function for 2b

In [641]:
# spot_free_interest_rates_after_given_dates

In [642]:
# list_of_future_expiry_dates

In [643]:
# What needs to be done
# Input: Spot Price, risk-free rate, Dividend yield, Time to Maturity of a contract

# Operations:

# since this HW2 was originally due on 30 September 2025, let's set this as our base date
# We need to also calculate our time to maturity (T) we do so by

# (future_expiry_date - base_date)/360 to follow an Act/360 day count convention
# As such, we assume a ACT/360 day-count convention as suggested here: https://www.cmegroup.com/articles/2024/index-options-box-spreads-as-financing-tool.html

# Implement formula of $$F_0 = S_0 * e^{(r-q)T}$$
# Return the Fair value of the futures relative to the base date

# Output: 
# Futures fair value

# ---------------
# # Set up base date
# base_date = datetime.datetime(2025, 9, 30)

# # Spot price of the underlying index today.
# S_0 = 6600.35
# r = 0.03637
# q = 0.011

# specific_future_expiry_date = datetime.datetime(int(list_of_future_expiry_dates[0][:4]), int(list_of_future_expiry_dates[0][5:7]), int(list_of_future_expiry_dates[0][8:10]))
# # print(base_date, specific_future_expiry_date)

# # Compute T value based on Act/360 day count convention where T is in terms of years
# difference = specific_future_expiry_date-base_date
# T = (difference.days + difference.seconds/86400)/360
# print(T)

# # Finally we shall implement the formula of $$F_0 = S_0 * e^{(r-q)T}$$
# F_0 = S_0 * math.exp((r-q)*T)
# print(F_0)

### 2c. Write a python function to calculate the futures fair value, assuming interest rate curve is static


In [645]:
# What needs to be done
# Input: Spot Price, risk-free rate, Dividend yield, Time to Maturity of a contract

# Operations:

# since this HW2 was originally due on 30 September 2025, let's set this as our base date
# We need to also calculate our time to maturity (T) we do so by

# (future_expiry_date - base_date)/360 to follow an Act/360 day count convention
# As such, we assume a ACT/360 day-count convention as suggested here: https://www.cmegroup.com/articles/2024/index-options-box-spreads-as-financing-tool.html

# Implement formula of $$F_0 = S_0 * e^{(r-q)T}$$
# Return the Fair value of the futures relative to the base date

# Output: 
# Futures fair value

# ---------------
# Set up base date
def futures_fair_value(S_0=6600.35, r=0.03637, q=0.011, inputed_date=list_of_future_expiry_dates[0]):
    base_date = datetime.datetime(2025, 9, 30)

    specific_future_expiry_date = datetime.datetime(int(inputed_date[:4]), int(inputed_date[5:7]), int(inputed_date[8:10]))

    # Compute T value based on Act/360 day count convention where T is in terms of years
    difference = specific_future_expiry_date-base_date
    T = (difference.days + difference.seconds/86400)/360

    # Finally we shall implement the formula of $$F_0 = S_0 * e^{(r-q)T}$$
    F_0 = S_0 * math.exp((r-q)*T)
    
    return F_0

To test our function

In [647]:
futures_fair_value()

6637.666398496328

Ok looks like our function works.

## 3. **Output results:**
   Calculate and output futures fair values given the spot index SPX=6600.35 and NDX=24223.69 in a table in the follow format

   | | DividendYld | SpotRate | SPX | NDX |
|---|---|---|---|---|
| Spot | | | 6600.35 | 24223.69 |
| Dec25 | | | | |
| Mar26 | | | | |
| Jun26 | | | | |
| Sep26 | | | | |
| Dec26 | | | | |

2. **Programming:**
    1. Write a python function to auto-generate the futures expiry dates
    2. Write a python function to look-up the spot risk free interest rate that has nearest maturity after a given date
    3. Write a python function to calculate the futures fair value, assuming interest rate curve is static

* First I use the python function to auto-generate the futures expiry dates

In [652]:
tuple_of_inputs = ('Dec25', 'Mar26', 'Jun26', 'Sep26', 'Dec26')
list_of_future_expiry_dates = []
for i in tuple_of_inputs:
    month = int(monthToNum(i[:3]))
    year = int('20'+i[3:])

    list_of_future_expiry_dates.append(third_friday(year, month))

list_of_future_expiry_dates

['2025-12-19', '2026-03-20', '2026-06-19', '2026-09-18', '2026-12-18']

* Then I use the function to look-up the spot risk free interest rate that has nearest maturity after a given date for the future expiry dates.

In [654]:
spot_free_interest_rates_after_given_dates = list(map(look_up_spot_rf_rate, list_of_future_expiry_dates))
spot_free_interest_rates_after_given_dates

[3.637, 3.6685, 3.541, 3.645, 3.5535]

* Now I use the python function to calculate the futures fair value, assuming interest rate curve is static

**Note:**
* Net Dividend Yield for SPX is 1.1%
* Net Dividend Yield for QQQ is 0.46%


In [656]:
tuple_of_inputs = ('Dec25', 'Mar26', 'Jun26', 'Sep26', 'Dec26')

list_of_future_fair_value_SPX = []
list_of_future_fair_value_NDX = []

initial_spot_value_SPX = 6600.35
initial_spot_value_NDX = 24223.69

dividend_yield_SPX = 0.011
dividend_yield_NDX = 0.0046

iteration_no = 0

for i in tuple_of_inputs:
    
    future_fair_value_SPX= futures_fair_value(initial_spot_value_SPX, 
                                               spot_free_interest_rates_after_given_dates[iteration_no]/100, 
                                               dividend_yield_SPX, 
                                               list_of_future_expiry_dates[iteration_no])

    future_fair_value_NDX= futures_fair_value(initial_spot_value_NDX, 
                                               spot_free_interest_rates_after_given_dates[iteration_no]/100, 
                                               dividend_yield_NDX, 
                                               list_of_future_expiry_dates[iteration_no])

    list_of_future_fair_value_SPX.append(round(future_fair_value_SPX,2))
    list_of_future_fair_value_NDX.append(round(future_fair_value_NDX, 2))
    iteration_no += 1

print(list_of_future_fair_value_SPX)
print(list_of_future_fair_value_NDX)

[6637.67, 6681.37, 6718.65, 6767.14, 6803.13]
[24395.31, 24595.7, 24772.99, 24992.15, 25165.76]


**Requested format**: Calculate and output futures fair values given the spot index SPX=6600.35 and NDX=24223.69 in a table in the follow format

| | DividendYld | SpotRate | SPX | NDX |
|---|---|---|---|---|
| Spot | | | 6600.35 | 24223.69 |
| Dec25 | | | | |
| Mar26 | | | | |
| Jun26 | | | | |
| Sep26 | | | | |
| Dec26 | | | | |

**My proposition**: Personally I do not agree with this table format. I believe than SPX and NDX indexes should be in separate tables in a format as follows.

| Index | Expiry Date | Fair Value | Risk-free Rate Use(%) | Dividend Yield Used (%) |
|---|---|---|---|---|
| SPX | | | | |
| SPX | | | | |
| SPX | | | | |
| SPX | | | | |
| SPX | | | | |

In [659]:
SPX_index_list = ["SPX"] * 5
SPX_dividend_yield = [1.1] * 5


SPX_df = pd.DataFrame({'Index': SPX_index_list, 'Expiry Date': list_of_future_expiry_dates, 'Fair Value': list_of_future_fair_value_SPX, 'Risk-free Rate Use(%)': spot_free_interest_rates_after_given_dates, 'Dividend Yield Used (%)': SPX_dividend_yield})
SPX_df

,Index,Expiry Date,Fair Value,Risk-free Rate Use(%),Dividend Yield Used (%)
0,SPX,2025-12-19,6637.67,3.6370,1.1
1,SPX,2026-03-20,6681.37,3.6685,1.1
2,SPX,2026-06-19,6718.65,3.5410,1.1
3,SPX,2026-09-18,6767.14,3.6450,1.1
4,SPX,2026-12-18,6803.13,3.5535,1.1


In [661]:
NDX_index_list = ["NDX"] * 5
NDX_dividend_yield = [0.46] * 5


NDX_df = pd.DataFrame({'Index': NDX_index_list, 'Expiry Date': list_of_future_expiry_dates, 'Fair Value': list_of_future_fair_value_NDX, 'Risk-free Rate Use(%)': spot_free_interest_rates_after_given_dates, 'Dividend Yield Used (%)': NDX_dividend_yield})
NDX_df

,Index,Expiry Date,Fair Value,Risk-free Rate Use(%),Dividend Yield Used (%)
0,NDX,2025-12-19,24395.31,3.6370,0.46
1,NDX,2026-03-20,24595.70,3.6685,0.46
2,NDX,2026-06-19,24772.99,3.5410,0.46
3,NDX,2026-09-18,24992.15,3.6450,0.46
4,NDX,2026-12-18,25165.76,3.5535,0.46
